In [1]:
# تنفيذ خوارزمية LIGHT10 (Light Stemming) على بيانات عربية
# باستخدام نفس البيانات التي استُخدمت مع Khoja

from datasets import load_dataset
import re

# =========================
# 1) تحميل البيانات (Wiki40B Arabic)
# =========================
dataset = load_dataset("wiki40b", "ar", split="train[:1%]")
texts = [x["text"] for x in dataset]

# =========================
# 2) تنظيف النص العربي
# =========================
ARABIC_DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652]")

def normalize_arabic(text):
    text = re.sub(ARABIC_DIACRITICS, "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[^ء-ي\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# =========================
# 3) تعريف خوارزمية LIGHT10
# =========================
# تعتمد على حذف السوابق واللواحق فقط (بدون استخراج جذر مثل Khoja)

PREFIXES = [
    "ال", "وال", "بال", "كال", "فال", "لل",
    "و", "ف", "ب", "ك", "ل", "س"
]

SUFFIXES = [
    "ه", "ها", "ك", "ي", "كم", "نا",
    "هم", "هن", "ا", "ان", "ون", "ين",
    "ات", "ة"
]


def light10_stem(word):
    # إزالة السوابق
    for p in PREFIXES:
        if word.startswith(p) and len(word) > len(p) + 2:
            word = word[len(p):]
            break

    # إزالة اللواحق
    for s in SUFFIXES:
        if word.endswith(s) and len(word) > len(s) + 2:
            word = word[:-len(s)]
            break

    return word

# =========================
# 4) تطبيق LIGHT10 على النصوص
# =========================

def apply_light10(text):
    text = normalize_arabic(text)
    tokens = text.split()
    stems = [light10_stem(tok) for tok in tokens]
    return stems

# تجربة على أول نص
sample_stems = apply_light10(texts[0])
print("عدد الكلمات بعد LIGHT10:", len(sample_stems))
print(sample_stems[:30])

# =========================
# 5) مقارنة سريعة: Khoja vs LIGHT10 (مفهومياً)
# =========================
# Khoja: استخراج جذر ثلاثي/رباعي (تحليل صرفي عميق)
# LIGHT10: حذف سوابق/لواحق فقط (سريع – مناسب للتصنيف)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: d9ae7f03-5130-42fb-8451-4bec25ecafff)')' thrown while requesting HEAD https://huggingface.co/datasets/google/wiki40b/resolve/main/README.md
Retrying in 1s [Retry 1/5].


عدد الكلمات بعد LIGHT10: 212
['نظري', 'معرفي', 'اجتماعي', 'اخلاق', 'تركز', 'نظري', 'معرفي', 'اجتماعي', 'اخلاق', 'علي', 'تمييز', 'بين', 'قدر', 'طفل', 'اخلاقي', 'اداء', 'اخلاق', 'له', 'تعتمد', 'كفاء', 'اخلاقي', 'او', 'امتلا', 'معارف', 'اخلاقي', 'شكل', 'رييس', 'علي', 'عملي', 'حسي']


In [ ]:
# Khoja

# استخراج جذر ثلاثي/رباعي

# تحليل صرفي عميق

# أدق لغويًا لكن أبطأ وأحيانًا يبالغ في التجذير

# LIGHT10

# Light Stemming (حذف سوابق + لواحق فقط)

# لا يحاول إيجاد الجذر

# ممتاز للتصنيف، clustering، TF-IDF

# 🔹 ما الذي طبقناه في الكود؟